# SERPAVI (2011-2023): limpieza y preparación

Objetivo: dejar la base en formato utilizable para análisis y modelo de alquiler.

Pasos:
- Cargar Excel y elegir la hoja útil.
- Estandarizar columnas.
- Convertir de formato wide a long.
- Filtrar Cantabria y Santa Cruz de Bezana.
- Calcular métricas rápidas para contraste.


In [184]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 180)


def find_project_root(start: Path | None = None) -> Path:
    # Detecta raíz por estructura del repo
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'notebooks').exists():
            return p
    return start


PROJECT_ROOT = find_project_root()
SERPAVI_DIR = PROJECT_ROOT / 'data' / 'raw' / 'MIVAU' / 'datos_alquiler'
DEFAULT_FILE = SERPAVI_DIR / '2025-09-10_bd_SERPAVI_2011-2023.xlsx'

if DEFAULT_FILE.exists():
    DATA_PATH = DEFAULT_FILE
else:
    candidates = sorted(SERPAVI_DIR.glob('*SERPAVI*.xlsx'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No encuentro ficheros SERPAVI en: {SERPAVI_DIR}')
    DATA_PATH = candidates[0]


print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_PATH: {DATA_PATH}')
DATA_PATH

PROJECT_ROOT: C:\Users\pdelr\Documents\MBA\BezanillaSL
DATA_PATH: C:\Users\pdelr\Documents\MBA\BezanillaSL\data\raw\MIVAU\datos_alquiler\2025-09-10_bd_SERPAVI_2011-2023.xlsx


WindowsPath('C:/Users/pdelr/Documents/MBA/BezanillaSL/data/raw/MIVAU/datos_alquiler/2025-09-10_bd_SERPAVI_2011-2023.xlsx')

In [185]:
# 1) Inspección de hojas
xls = pd.ExcelFile(DATA_PATH)
sheet_names = xls.sheet_names

print(f'Total hojas: {len(sheet_names)}')
for i, s in enumerate(sheet_names, start=1):
    print(f'{i:02d}. {s}')

Total hojas: 6
01. Metadatos
02. CCAA
03. Provincias
04. Municipios
05. Distritos
06. Secciones censales


In [186]:
# 2) Carga de hojas para revisar estructura
dfs_raw = {}
for s in sheet_names:
    try:
        dfs_raw[s] = pd.read_excel(DATA_PATH, sheet_name=s)
    except Exception as e:
        print(f'[WARN] No pude leer {s}: {e}')

summary_sheets = pd.DataFrame([
    {'hoja': k, 'filas': v.shape[0], 'columnas': v.shape[1]}
    for k, v in dfs_raw.items()
]).sort_values(['columnas', 'filas'], ascending=False)

print(f'Hojas cargadas: {len(dfs_raw)} / {len(sheet_names)}')
print(summary_sheets.to_string(index=False))

Hojas cargadas: 6 / 6
              hoja  filas  columnas
Secciones censales  35122       265
         Distritos  10280       265
        Municipios   7934       264
        Provincias     50       262
              CCAA     24       262
         Metadatos     39         3


In [187]:
# 3) Vista rápida de la primera hoja
first_sheet = sheet_names[0]
dfs_raw[first_sheet].head(10)

,Sistema Estatal Índices Alquiler de Vivienda,Unnamed: 1,Unnamed: 2
0,(Explotación estadística de fuentes tributarias),NaN,NaN
1,Datos rentas y arrendamientos Modelo 100 por a...,NaN,NaN
2,Diseño de registro,NaN,NaN
3,Código,Denominación / definición,Tipo
4,CUSEC_R,Código único de Sección Censal Censo Población...,VARCHAR(10)
5,CUSEC,Código único de Sección Censal Censo Población...,INT
6,CUMUN,Código único de Municipio Censo Población 2011...,INT
7,CSEC,Código de Sección Censal Censo Población 2011 ...,INT
8,CDIS,Código de Distrito Censal Censo Población 2011...,INT
9,CMUN,Código Municipio Censo Población 2011 INE (num...,INT


In [188]:
# 4) Vista rápida de la última hoja
last_sheet = sheet_names[-1]
dfs_raw[last_sheet].head(10)

,CPRO,LITPRO,CUMUN,LITMUN,CUSEC,BI_ALVHEPCO_TVC_11,BI_ALVHEPCO_TVU_11,ALQM2_LV_M_VC_11,ALQM2_LV_25_VC_11,ALQM2_LV_75_VC_11,ALQM2_LV_M_VU_11,ALQM2_LV_25_VU_11,ALQM2_LV_75_VU_11,ALQTBID12_M_VC_11,ALQTBID12_25_VC_11,ALQTBID12_75_VC_11,ALQTBID12_M_VU_11,ALQTBID12_25_VU_11,ALQTBID12_75_VU_11,SLVM2_M_VC_11,SLVM2_25_VC_11,SLVM2_75_VC_11,SLVM2_M_VU_11,SLVM2_25_VU_11,SLVM2_75_VU_11,BI_ALVHEPCO_TVC_12,BI_ALVHEPCO_TVU_12,ALQM2_LV_M_VC_12,ALQM2_LV_25_VC_12,ALQM2_LV_75_VC_12,ALQM2_LV_M_VU_12,ALQM2_LV_25_VU_12,ALQM2_LV_75_VU_12,ALQTBID12_M_VC_12,ALQTBID12_25_VC_12,ALQTBID12_75_VC_12,ALQTBID12_M_VU_12,ALQTBID12_25_VU_12,ALQTBID12_75_VU_12,SLVM2_M_VC_12,SLVM2_25_VC_12,SLVM2_75_VC_12,SLVM2_M_VU_12,SLVM2_25_VU_12,SLVM2_75_VU_12,BI_ALVHEPCO_TVC_13,BI_ALVHEPCO_TVU_13,ALQM2_LV_M_VC_13,ALQM2_LV_25_VC_13,ALQM2_LV_75_VC_13,ALQM2_LV_M_VU_13,ALQM2_LV_25_VU_13,ALQM2_LV_75_VU_13,ALQTBID12_M_VC_13,ALQTBID12_25_VC_13,ALQTBID12_75_VC_13,ALQTBID12_M_VU_13,ALQTBID12_25_VU_13,ALQTBID12_75_VU_13,SLVM2_M_VC_13,SLVM2_25_VC_13,SLVM2_75_VC_13,SLVM2_M_VU_13,SLVM2_25_VU_13,SLVM2_75_VU_13,BI_ALVHEPCO_TVC_14,BI_ALVHEPCO_TVU_14,ALQM2_LV_M_VC_14,ALQM2_LV_25_VC_14,ALQM2_LV_75_VC_14,ALQM2_LV_M_VU_14,ALQM2_LV_25_VU_14,ALQM2_LV_75_VU_14,ALQTBID12_M_VC_14,ALQTBID12_25_VC_14,ALQTBID12_75_VC_14,ALQTBID12_M_VU_14,ALQTBID12_25_VU_14,ALQTBID12_75_VU_14,SLVM2_M_VC_14,SLVM2_25_VC_14,SLVM2_75_VC_14,SLVM2_M_VU_14,SLVM2_25_VU_14,SLVM2_75_VU_14,BI_ALVHEPCO_TVC_15,BI_ALVHEPCO_TVU_15,ALQM2_LV_M_VC_15,ALQM2_LV_25_VC_15,ALQM2_LV_75_VC_15,ALQM2_LV_M_VU_15,ALQM2_LV_25_VU_15,ALQM2_LV_75_VU_15,ALQTBID12_M_VC_15,ALQTBID12_25_VC_15,ALQTBID12_75_VC_15,ALQTBID12_M_VU_15,ALQTBID12_25_VU_15,ALQTBID12_75_VU_15,SLVM2_M_VC_15,SLVM2_25_VC_15,SLVM2_75_VC_15,SLVM2_M_VU_15,SLVM2_25_VU_15,SLVM2_75_VU_15,BI_ALVHEPCO_TVC_16,BI_ALVHEPCO_TVU_16,ALQM2_LV_M_VC_16,ALQM2_LV_25_VC_16,ALQM2_LV_75_VC_16,...,ALQTBID12_75_VC_18,ALQTBID12_M_VU_18,ALQTBID12_25_VU_18,ALQTBID12_75_VU_18,SLVM2_M_VC_18,SLVM2_25_VC_18,SLVM2_75_VC_18,SLVM2_M_VU_18,SLVM2_25_VU_18,SLVM2_75_VU_18,BI_ALVHEPCO_TVC_19,BI_ALVHEPCO_TVU_19,ALQM2_LV_M_VC_19,ALQM2_LV_25_VC_19,ALQM2_LV_75_VC_19,ALQM2_LV_M_VU_19,ALQM2_LV_25_VU_19,ALQM2_LV_75_VU_19,ALQTBID12_M_VC_19,ALQTBID12_25_VC_19,ALQTBID12_75_VC_19,ALQTBID12_M_VU_19,ALQTBID12_25_VU_19,ALQTBID12_75_VU_19,SLVM2_M_VC_19,SLVM2_25_VC_19,SLVM2_75_VC_19,SLVM2_M_VU_19,SLVM2_25_VU_19,SLVM2_75_VU_19,BI_ALVHEPCO_TVC_20,BI_ALVHEPCO_TVU_20,ALQM2_LV_M_VC_20,ALQM2_LV_25_VC_20,ALQM2_LV_75_VC_20,ALQM2_LV_M_VU_20,ALQM2_LV_25_VU_20,ALQM2_LV_75_VU_20,ALQTBID12_M_VC_20,ALQTBID12_25_VC_20,ALQTBID12_75_VC_20,ALQTBID12_M_VU_20,ALQTBID12_25_VU_20,ALQTBID12_75_VU_20,SLVM2_M_VC_20,SLVM2_25_VC_20,SLVM2_75_VC_20,SLVM2_M_VU_20,SLVM2_25_VU_20,SLVM2_75_VU_20,BI_ALVHEPCO_TVC_21,BI_ALVHEPCO_TVU_21,ALQM2_LV_M_VC_21,ALQM2_LV_25_VC_21,ALQM2_LV_75_VC_21,ALQM2_LV_M_VU_21,ALQM2_LV_25_VU_21,ALQM2_LV_75_VU_21,ALQTBID12_M_VC_21,ALQTBID12_25_VC_21,ALQTBID12_75_VC_21,ALQTBID12_M_VU_21,ALQTBID12_25_VU_21,ALQTBID12_75_VU_21,SLVM2_M_VC_21,SLVM2_25_VC_21,SLVM2_75_VC_21,SLVM2_M_VU_21,SLVM2_25_VU_21,SLVM2_75_VU_21,BI_ALVHEPCO_TVC_22,BI_ALVHEPCO_TVU_22,ALQM2_LV_M_VC_22,ALQM2_LV_25_VC_22,ALQM2_LV_75_VC_22,ALQM2_LV_M_VU_22,ALQM2_LV_25_VU_22,ALQM2_LV_75_VU_22,ALQTBID12_M_VC_22,ALQTBID12_25_VC_22,ALQTBID12_75_VC_22,ALQTBID12_M_VU_22,ALQTBID12_25_VU_22,ALQTBID12_75_VU_22,SLVM2_M_VC_22,SLVM2_25_VC_22,SLVM2_75_VC_22,SLVM2_M_VU_22,SLVM2_25_VU_22,SLVM2_75_VU_22,BI_ALVHEPCO_TVC_23,BI_ALVHEPCO_TVU_23,ALQM2_LV_M_VC_23,ALQM2_LV_25_VC_23,ALQM2_LV_75_VC_23,ALQM2_LV_M_VU_23,ALQM2_LV_25_VU_23,ALQM2_LV_75_VU_23,ALQTBID12_M_VC_23,ALQTBID12_25_VC_23,ALQTBID12_75_VC_23,ALQTBID12_M_VU_23,ALQTBID12_25_VU_23,ALQTBID12_75_VU_23,SLVM2_M_VC_23,SLVM2_25_VC_23,SLVM2_75_VC_23,SLVM2_M_VU_23,SLVM2_25_VU_23,SLVM2_75_VU_23
0,2,Albacete,2001,Abengibre,200101001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [189]:
# 5) Nos quedamos únicamente con el dataset de distritos censales
df_distritos = dfs_raw[last_sheet].copy()

In [190]:
# 6) Renombre de columnas SERPAVI

def parse_year_suffix(y_txt: str) -> int:
    y = int(y_txt)
    return 2000 + y if len(y_txt) == 2 else y


def rename_serpavi_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    fixed = {
        'CPRO': 'codigo_provincia',
        'CMUN': 'codigo_municipio',
        'CUMUN': 'codigo_municipio_unico',
        'CUSEC': 'codigo_seccion_censal',
        'CUSEC_R': 'codigo_seccion_censal_str',
        'CSEC': 'codigo_seccion_censal_num',
        'NPRO': 'nombre_provincia',
        'LITPRO': 'nombre_provincia',
        'NCA': 'nombre_comunidad_autonoma',
        'NMUN': 'nombre_municipio',
        'LITMUN': 'nombre_municipio',
    }

    metric_map = {
        'BI_ALVHEPCO_TVC': 'n_contratos_vivienda_colectiva',
        'BI_ALVHEPCO_TVU': 'n_contratos_vivienda_unifamiliar',
        'ALQM2_LV_M_VC': 'alquiler_m2_mediana_vc',
        'ALQM2_LV_25_VC': 'alquiler_m2_p25_vc',
        'ALQM2_LV_75_VC': 'alquiler_m2_p75_vc',
        'ALQM2_LV_M_VU': 'alquiler_m2_mediana_vu',
        'ALQM2_LV_25_VU': 'alquiler_m2_p25_vu',
        'ALQM2_LV_75_VU': 'alquiler_m2_p75_vu',
        'ALQTBID12_M_VC': 'alquiler_total_mediana_vc',
        'ALQTBID12_25_VC': 'alquiler_total_p25_vc',
        'ALQTBID12_75_VC': 'alquiler_total_p75_vc',
        'ALQTBID12_M_VU': 'alquiler_total_mediana_vu',
        'ALQTBID12_25_VU': 'alquiler_total_p25_vu',
        'ALQTBID12_75_VU': 'alquiler_total_p75_vu',
        'SLVM2_M_VC': 'superficie_m2_mediana_vc',
        'SLVM2_25_VC': 'superficie_m2_p25_vc',
        'SLVM2_75_VC': 'superficie_m2_p75_vc',
        'SLVM2_M_VU': 'superficie_m2_mediana_vu',
        'SLVM2_25_VU': 'superficie_m2_p25_vu',
        'SLVM2_75_VU': 'superficie_m2_p75_vu',
    }

    new_cols = []
    for c in out.columns:
        c_str = str(c)

        if c_str in fixed:
            new_cols.append(fixed[c_str])
            continue

        m = re.match(r'^([A-Za-z0-9_]+)_(\d{2}|\d{4})$', c_str)
        if m:
            base, yy = m.groups()
            if base in metric_map:
                new_cols.append(f"{metric_map[base]}_{parse_year_suffix(yy)}")
                continue

        new_cols.append(norm_col(c_str))

    out.columns = new_cols

    if out.columns.duplicated().any():
        seen = {}
        dedup = []
        for c in out.columns:
            if c not in seen:
                seen[c] = 1
                dedup.append(c)
            else:
                seen[c] += 1
                dedup.append(f'{c}_{seen[c]}')
        out.columns = dedup

    return out


df_main = rename_serpavi_columns(df_distritos)

print(f'Total columnas: {len(df_main.columns)}')
print('Primeras 35 columnas:')
print(df_main.columns[:35])

Total columnas: 265
Primeras 35 columnas:
Index(['codigo_provincia', 'nombre_provincia', 'codigo_municipio_unico', 'nombre_municipio', 'codigo_seccion_censal', 'n_contratos_vivienda_colectiva_2011',
       'n_contratos_vivienda_unifamiliar_2011', 'alquiler_m2_mediana_vc_2011', 'alquiler_m2_p25_vc_2011', 'alquiler_m2_p75_vc_2011', 'alquiler_m2_mediana_vu_2011',
       'alquiler_m2_p25_vu_2011', 'alquiler_m2_p75_vu_2011', 'alquiler_total_mediana_vc_2011', 'alquiler_total_p25_vc_2011', 'alquiler_total_p75_vc_2011',
       'alquiler_total_mediana_vu_2011', 'alquiler_total_p25_vu_2011', 'alquiler_total_p75_vu_2011', 'superficie_m2_mediana_vc_2011', 'superficie_m2_p25_vc_2011',
       'superficie_m2_p75_vc_2011', 'superficie_m2_mediana_vu_2011', 'superficie_m2_p25_vu_2011', 'superficie_m2_p75_vu_2011', 'n_contratos_vivienda_colectiva_2012',
       'n_contratos_vivienda_unifamiliar_2012', 'alquiler_m2_mediana_vc_2012', 'alquiler_m2_p25_vc_2012', 'alquiler_m2_p75_vc_2012', 'alquiler_m2_mediana

## Validaciones mínimas

Chequeo rápido de nulos, tipos y duplicados antes de transformar.


In [191]:
# 7) Perfilado rápido
profile = pd.DataFrame({
    'columna': df_main.columns,
    'dtype': [str(df_main[c].dtype) for c in df_main.columns],
    'null_pct': [100 * df_main[c].isna().mean() for c in df_main.columns],
    'n_unicos': [df_main[c].nunique(dropna=True) for c in df_main.columns],
}).sort_values('null_pct', ascending=False)

print('Top 20 columnas con más nulos (%):')
print(profile.head(20).to_string(index=False))
print('\nTop 20 columnas con menos nulos (%):')
print(profile.sort_values('null_pct', ascending=True).head(20).to_string(index=False))


Top 20 columnas con más nulos (%):
                       columna   dtype  null_pct  n_unicos
     superficie_m2_p75_vu_2011 float64 89.849667      1222
     superficie_m2_p25_vu_2011 float64 89.849667       733
    alquiler_total_p25_vu_2011 float64 89.849667      2509
    alquiler_total_p75_vu_2011 float64 89.849667      2510
 superficie_m2_mediana_vu_2011 float64 89.849667       554
alquiler_total_mediana_vu_2011 float64 89.849667      2211
   alquiler_m2_mediana_vu_2011 float64 89.849667      3314
       alquiler_m2_p75_vu_2011 float64 89.849667      3437
       alquiler_m2_p25_vu_2011 float64 89.849667      3476
   alquiler_m2_mediana_vu_2012 float64 88.759182      3581
     superficie_m2_p25_vu_2012 float64 88.759182       749
       alquiler_m2_p75_vu_2012 float64 88.759182      3746
    alquiler_total_p25_vu_2012 float64 88.759182      2688
alquiler_total_mediana_vu_2012 float64 88.759182      2246
    alquiler_total_p75_vu_2012 float64 88.759182      2653
 superficie_m2_median

In [192]:
# 8) Duplicados exactos
dup_count = int(df_main.duplicated().sum())
print(f'Duplicados exactos: {dup_count:,}')


Duplicados exactos: 0


## Transformación wide a long

Convierto `metrica_YYYY` a una estructura con una fila por año y unidad geográfica.


In [193]:
# 9) Conversión wide -> long

def serpavi_wide_to_long(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    year_pat = re.compile(r'.*_\d{4}$')
    id_vars = [c for c in d.columns if not year_pat.match(c)]
    value_vars = [c for c in d.columns if year_pat.match(c)]

    if not value_vars:
        raise ValueError('No encontré columnas con sufijo _YYYY. Revisa el renombre previo.')

    long = d.melt(id_vars=id_vars, value_vars=value_vars, var_name='variable_anual', value_name='valor')
    long[['metrica', 'anio']] = long['variable_anual'].str.rsplit('_', n=1, expand=True)
    long['anio'] = pd.to_numeric(long['anio'], errors='coerce').astype('Int64')

    out = (
        long.drop(columns=['variable_anual'])
            .pivot_table(index=id_vars + ['anio'], columns='metrica', values='valor', aggfunc='first')
            .reset_index()
    )
    out.columns.name = None
    return out


df_long = serpavi_wide_to_long(df_main)

print(f'df_main (wide): {df_main.shape[0]:,} x {df_main.shape[1]:,}')
print(f'df_long (long): {df_long.shape[0]:,} x {df_long.shape[1]:,}')
print(df_long.head(5).to_string(index=False))

df_main (wide): 35,122 x 265
df_long (long): 414,492 x 26
 codigo_provincia nombre_provincia  codigo_municipio_unico nombre_municipio  codigo_seccion_censal  anio  alquiler_m2_mediana_vc  alquiler_m2_mediana_vu  alquiler_m2_p25_vc  alquiler_m2_p25_vu  alquiler_m2_p75_vc  alquiler_m2_p75_vu  alquiler_total_mediana_vc  alquiler_total_mediana_vu  alquiler_total_p25_vc  alquiler_total_p25_vu  alquiler_total_p75_vc  alquiler_total_p75_vu  n_contratos_vivienda_colectiva  n_contratos_vivienda_unifamiliar  superficie_m2_mediana_vc  superficie_m2_mediana_vu  superficie_m2_p25_vc  superficie_m2_p25_vu  superficie_m2_p75_vc  superficie_m2_p75_vu
                2         Albacete                    2001        Abengibre              200101001  2012                     NaN                     NaN                 NaN                 NaN                 NaN                 NaN                        NaN                        NaN                    NaN                    NaN                    NaN  

In [194]:
# 10) Comprobación de estructura long
print(f'Número de columnas: {len(df_long.columns)}')
print('Primeras 30 columnas:')
print(df_long.columns[:30].tolist())
df_long.shape, df_long.columns[:25].tolist()


Número de columnas: 26
Primeras 30 columnas:
['codigo_provincia', 'nombre_provincia', 'codigo_municipio_unico', 'nombre_municipio', 'codigo_seccion_censal', 'anio', 'alquiler_m2_mediana_vc', 'alquiler_m2_mediana_vu', 'alquiler_m2_p25_vc', 'alquiler_m2_p25_vu', 'alquiler_m2_p75_vc', 'alquiler_m2_p75_vu', 'alquiler_total_mediana_vc', 'alquiler_total_mediana_vu', 'alquiler_total_p25_vc', 'alquiler_total_p25_vu', 'alquiler_total_p75_vc', 'alquiler_total_p75_vu', 'n_contratos_vivienda_colectiva', 'n_contratos_vivienda_unifamiliar', 'superficie_m2_mediana_vc', 'superficie_m2_mediana_vu', 'superficie_m2_p25_vc', 'superficie_m2_p25_vu', 'superficie_m2_p75_vc', 'superficie_m2_p75_vu']


((414492, 26),
 ['codigo_provincia',
  'nombre_provincia',
  'codigo_municipio_unico',
  'nombre_municipio',
  'codigo_seccion_censal',
  'anio',
  'alquiler_m2_mediana_vc',
  'alquiler_m2_mediana_vu',
  'alquiler_m2_p25_vc',
  'alquiler_m2_p25_vu',
  'alquiler_m2_p75_vc',
  'alquiler_m2_p75_vu',
  'alquiler_total_mediana_vc',
  'alquiler_total_mediana_vu',
  'alquiler_total_p25_vc',
  'alquiler_total_p25_vu',
  'alquiler_total_p75_vc',
  'alquiler_total_p75_vu',
  'n_contratos_vivienda_colectiva',
  'n_contratos_vivienda_unifamiliar',
  'superficie_m2_mediana_vc',
  'superficie_m2_mediana_vu',
  'superficie_m2_p25_vc',
  'superficie_m2_p25_vu',
  'superficie_m2_p75_vc'])

## Filtros geográficos

Genero subconjuntos para Cantabria y Santa Cruz de Bezana.










In [195]:
# 11) Subconjuntos geográficos
df_cantabria = None
df_bezana = None

if 'codigo_provincia' in df_long.columns:
    prov_num = pd.to_numeric(df_long['codigo_provincia'], errors='coerce')
    df_cantabria = df_long[prov_num == 39].copy()
else:
    print('[WARN] No existe codigo_provincia; no se puede filtrar Cantabria por código.')

if 'nombre_municipio' in df_long.columns:
    mask_bezana = df_long['nombre_municipio'].astype(str).str.lower().str.contains('bezan', na=False)
    df_bezana = df_long[mask_bezana].copy()
elif 'codigo_municipio' in df_long.columns:
    mun_num = pd.to_numeric(df_long['codigo_municipio'], errors='coerce')
    df_bezana = df_long[mun_num == 73].copy()
else:
    print('[WARN] No existe nombre_municipio ni codigo_municipio; no se puede filtrar Bezana.')

print(f'Cantabria: {None if df_cantabria is None else df_cantabria.shape}')
print(f'Bezana: {None if df_bezana is None else df_bezana.shape}')


Cantabria: (5938, 26)
Bezana: (104, 26)


## Métricas para contraste rápido
Me centro en columnas de alquiler, contratos y superficie.

In [196]:
# 12) Columnas de interés
metric_cols = [
    c for c in df_long.columns
    if any(k in c for k in ['alquiler_m2_', 'alquiler_total_', 'n_contratos_', 'superficie_m2_'])
]

print(f'Total métricas: {len(metric_cols)}')
print(metric_cols[:40])
metric_cols[:30]

Total métricas: 20
['alquiler_m2_mediana_vc', 'alquiler_m2_mediana_vu', 'alquiler_m2_p25_vc', 'alquiler_m2_p25_vu', 'alquiler_m2_p75_vc', 'alquiler_m2_p75_vu', 'alquiler_total_mediana_vc', 'alquiler_total_mediana_vu', 'alquiler_total_p25_vc', 'alquiler_total_p25_vu', 'alquiler_total_p75_vc', 'alquiler_total_p75_vu', 'n_contratos_vivienda_colectiva', 'n_contratos_vivienda_unifamiliar', 'superficie_m2_mediana_vc', 'superficie_m2_mediana_vu', 'superficie_m2_p25_vc', 'superficie_m2_p25_vu', 'superficie_m2_p75_vc', 'superficie_m2_p75_vu']


['alquiler_m2_mediana_vc',
 'alquiler_m2_mediana_vu',
 'alquiler_m2_p25_vc',
 'alquiler_m2_p25_vu',
 'alquiler_m2_p75_vc',
 'alquiler_m2_p75_vu',
 'alquiler_total_mediana_vc',
 'alquiler_total_mediana_vu',
 'alquiler_total_p25_vc',
 'alquiler_total_p25_vu',
 'alquiler_total_p75_vc',
 'alquiler_total_p75_vu',
 'n_contratos_vivienda_colectiva',
 'n_contratos_vivienda_unifamiliar',
 'superficie_m2_mediana_vc',
 'superficie_m2_mediana_vu',
 'superficie_m2_p25_vc',
 'superficie_m2_p25_vu',
 'superficie_m2_p75_vc',
 'superficie_m2_p75_vu']

In [197]:
# 13) Agregación anual de ejemplo

def agg_by_year(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    d = df.dropna(subset=['anio', value_col]).copy()
    return d.groupby('anio')[value_col].agg(['count', 'mean', 'median']).reset_index()

example_col = next((c for c in ['alquiler_m2_mediana_vc', 'alquiler_m2_mediana_vu'] if c in df_long.columns), None)

if example_col is None:
    print('No encuentro alquiler_m2_mediana_vc/vu. Elige una métrica manualmente.')
else:
    print(f'Columna usada: {example_col}')
    if df_cantabria is not None and len(df_cantabria):
        print('\nCantabria (primeras 10 filas):')
        print(agg_by_year(df_cantabria, example_col).head(10).to_string(index=False))
    if df_bezana is not None and len(df_bezana):
        print('\nBezana (primeras 10 filas):')
        print(agg_by_year(df_bezana, example_col).head(10).to_string(index=False))

Columna usada: alquiler_m2_mediana_vc

Cantabria (primeras 10 filas):
 anio  count     mean   median
 2011    320 5.717259 5.867838
 2012    322 5.659255 5.778060
 2013    325 5.405696 5.514706
 2014    330 5.316269 5.318960
 2015    335 5.280416 5.223881
 2016    347 5.331767 5.361328
 2017    353 5.416181 5.379721
 2018    357 5.519262 5.555556
 2019    362 5.850841 5.823643
 2020    367 5.963169 5.904118

Bezana (primeras 10 filas):
 anio  count     mean   median
 2011      6 6.294309 6.305765
 2012      7 6.207436 6.321839
 2013      7 5.729563 5.725490
 2014      7 5.712756 5.487805
 2015      7 5.533701 5.540357
 2016      7 5.682390 5.555556
 2017      7 5.753611 5.813953
 2018      7 5.872770 5.791592
 2019      7 6.123980 5.875000
 2020      7 6.309463 6.039960


In [198]:
# 14) Crecimiento acumulado y CAGR

def rent_growth_metrics(df: pd.DataFrame | None, col: str):
    if df is None or len(df) == 0 or col not in df.columns:
        return None

    s = (
        df.dropna(subset=['anio', col])
          .groupby('anio')[col]
          .median()
          .sort_index()
    )

    if len(s) < 2:
        return None

    growth_total = (s.iloc[-1] / s.iloc[0]) - 1
    years = int(s.index.max() - s.index.min())
    cagr = (s.iloc[-1] / s.iloc[0]) ** (1 / years) - 1 if years > 0 else None

    return {
        'first_year': int(s.index.min()),
        'last_year': int(s.index.max()),
        'growth_total_pct': round(float(growth_total) * 100, 2),
        'cagr_pct': round(float(cagr) * 100, 2) if cagr is not None else None,
    }

growth_bezana = rent_growth_metrics(df_bezana, 'alquiler_m2_mediana_vc')
growth_cantabria = rent_growth_metrics(df_cantabria, 'alquiler_m2_mediana_vc')

print(f'Bezana: {growth_bezana}')
print(f'Cantabria: {growth_cantabria}')

Bezana: {'first_year': 2011, 'last_year': 2023, 'growth_total_pct': 9.53, 'cagr_pct': 0.76}
Cantabria: {'first_year': 2011, 'last_year': 2023, 'growth_total_pct': 12.84, 'cagr_pct': 1.01}


In [199]:
# 15) Crecimiento desde 2018

def growth_since(df: pd.DataFrame | None, col: str, start_year: int = 2018):
    if df is None or len(df) == 0 or col not in df.columns:
        return None

    s = (
        df[df['anio'] >= start_year]
        .dropna(subset=[col])
        .groupby('anio')[col]
        .median()
        .sort_index()
    )

    if len(s) < 2:
        return None

    growth = (s.iloc[-1] / s.iloc[0]) - 1
    return round(growth * 100, 2)

growth_2018_bezana = growth_since(df_bezana, 'alquiler_m2_mediana_vc', 2018)
growth_2018_cant = growth_since(df_cantabria, 'alquiler_m2_mediana_vc', 2018)

print(f'Bezana: {growth_2018_bezana}%')
print(f'Cantabria: {growth_2018_cant}%')

Bezana: 19.25%
Cantabria: 19.18%


In [200]:
# 16) Tendencia de contratos

def contracts_trend(df: pd.DataFrame | None, col: str) -> pd.Series:
    if df is None or len(df) == 0 or col not in df.columns:
        return pd.Series(dtype='float64')

    return (
        df.dropna(subset=['anio', col])
          .groupby('anio')[col]
          .sum()
          .sort_index()
    )

contracts_bezana = contracts_trend(df_bezana, 'n_contratos_vivienda_colectiva')
contracts_cantabria = contracts_trend(df_cantabria, 'n_contratos_vivienda_colectiva')

print(f'Años con dato (Bezana): {len(contracts_bezana)}')
print(f'Años con dato (Cantabria): {len(contracts_cantabria)}')
print(contracts_bezana.head(13))

Años con dato (Bezana): 13
Años con dato (Cantabria): 13
anio
2011    295.0
2012    306.0
2013    313.0
2014    338.0
2015    382.0
2016    416.0
2017    436.0
2018    448.0
2019    458.0
2020    471.0
2021    497.0
2022    508.0
2023    520.0
Name: n_contratos_vivienda_colectiva, dtype: float64


In [201]:
# 17) Nivel relativo de renta (Bezana / Cantabria)
if df_bezana is not None and len(df_bezana) and 'alquiler_m2_mediana_vc' in df_bezana.columns:
    rent_bezana = df_bezana.groupby('anio')['alquiler_m2_mediana_vc'].median()
else:
    rent_bezana = pd.Series(dtype='float64')

if df_cantabria is not None and len(df_cantabria) and 'alquiler_m2_mediana_vc' in df_cantabria.columns:
    rent_cantabria = df_cantabria.groupby('anio')['alquiler_m2_mediana_vc'].median()
else:
    rent_cantabria = pd.Series(dtype='float64')

relative_ratio = (rent_bezana / rent_cantabria).dropna()

if len(relative_ratio):
    print(f'Ratio medio Bezana/Cantabria: {relative_ratio.mean():.3f}')
    print('Últimos valores del ratio:')
    print(relative_ratio.tail().to_string())
else:
    print('No hay datos suficientes para calcular el ratio.')

Ratio medio Bezana/Cantabria: 1.046
Últimos valores del ratio:
anio
2019    1.008819
2020    1.023008
2021    1.029605
2022    1.032878
2023    1.043069


In [202]:
# 18) Score simple de tensión
score = 0

if growth_bezana and growth_bezana.get('growth_total_pct') is not None and growth_bezana['growth_total_pct'] > 15:
    score += 1
if growth_2018_bezana is not None and growth_2018_bezana > 8:
    score += 1
if len(relative_ratio) and relative_ratio.mean() >= 1:
    score += 1

print(f'Score final: {score} (rango 0-3)')

Score final: 2 (rango 0-3)


In [203]:
# 19) Traducción de score a theta base
if score == 3:
    theta_base = 0.95
elif score == 2:
    theta_base = 0.93
elif score == 1:
    theta_base = 0.91
else:
    theta_base = 0.90

print(f'Theta base: {theta_base:.2f}')

Theta base: 0.93


In [204]:
# 20) Resumen final
summary = {
    'growth_total_pct_bezana': growth_bezana,
    'growth_2018_to_last_pct_bezana': growth_2018_bezana,
    'relative_rent_ratio_mean': round(relative_ratio.mean(), 3) if len(relative_ratio) else None,
    'suggested_theta_base': theta_base,
}

for k, v in summary.items():
    print(f'- {k}: {v}')

- growth_total_pct_bezana: {'first_year': 2011, 'last_year': 2023, 'growth_total_pct': 9.53, 'cagr_pct': 0.76}
- growth_2018_to_last_pct_bezana: 19.25
- relative_rent_ratio_mean: 1.046
- suggested_theta_base: 0.93
